# SniffTest Training Notebook

This notebook showcases the training loop to be executed. It abstracts away the complexity of the training process such that the primary focus is on the env performance and evals during and post training.

Expected flow:
1. Install training dependencies
2. Optional sanity check
3. Export SFT examples
4. Run SFT
5. Run RL from the SFT checkpoint
6. Evaluate baseline vs post-SFT vs post-RL
7. Render the combined reward/eval report


## Runtime Notes

- Re-running the cells is safe: commands overwrite or refresh the expected artifacts in `training_outputs/`.


In [1]:
from __future__ import annotations

import json
import os
import shlex
import subprocess
from pathlib import Path

from IPython.display import HTML, Image, display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "training" / "train.py").exists() and (candidate / "README.md").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing training/train.py and README.md")


ROOT = find_repo_root(Path.cwd()).resolve()
os.chdir(ROOT)

TRAIN_OUTPUT_DIR = ROOT / "training_outputs"
SFT_DIR = TRAIN_OUTPUT_DIR / "sft_warm_start"
FINAL_DIR = TRAIN_OUTPUT_DIR / "final_merged"
REWARD_HISTORY = TRAIN_OUTPUT_DIR / "reward_history.json"
EVAL_METRICS = TRAIN_OUTPUT_DIR / "test_eval_metrics.json"
REPORT_IMAGE = TRAIN_OUTPUT_DIR / "training_and_eval_report.png"

print(f"Repo root: {ROOT}")
print(f"Training outputs: {TRAIN_OUTPUT_DIR}")


def run(cmd: str) -> None:
    print(f"\\n$ {cmd}\\n")
    process = subprocess.Popen(
        shlex.split(cmd),
        cwd=ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    exit_code = process.wait()
    if exit_code != 0:
        raise RuntimeError(f"Command failed with exit code {exit_code}: {cmd}")


def load_json(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Expected artifact not found: {path}")
    return json.loads(path.read_text())


def show_eval_table(path: Path) -> None:
    payload = load_json(path)
    rows = []
    for model in payload.get("models", []):
        summary = model.get("summary", {})
        rows.append(
            "<tr>"
            f"<td>{summary.get('model_label', '')}</td>"
            f"<td>{summary.get('json_valid_rate', '')}</td>"
            f"<td>{summary.get('verdict_accuracy', '')}</td>"
            f"<td>{summary.get('avg_steps_to_verdict', '')}</td>"
            f"<td>{summary.get('tool_use_rate', '')}</td>"
            f"<td>{summary.get('avg_total_reward', '')}</td>"
            "</tr>"
        )

    html = """
    <table>
      <thead>
        <tr>
          <th>model</th>
          <th>json_valid_rate</th>
          <th>verdict_accuracy</th>
          <th>avg_steps_to_verdict</th>
          <th>tool_use_rate</th>
          <th>avg_total_reward</th>
        </tr>
      </thead>
      <tbody>
    """ + "".join(rows) + """
      </tbody>
    </table>
    """
    display(HTML(html))


def show_reward_summary(path: Path) -> None:
    history = load_json(path)
    print(f"Episodes logged: {len(history)}")
    by_level = {}
    for row in history:
        by_level.setdefault(row['level'], []).append(row['reward'])
    for level, rewards in by_level.items():
        avg_reward = sum(rewards) / max(len(rewards), 1)
        print(f"{level}: n={len(rewards)} avg_reward={avg_reward:.4f} last_reward={rewards[-1]:.4f}")


Repo root: /Users/kevin/Desktop/snifftest_env
Training outputs: /Users/kevin/Desktop/snifftest_env/training_outputs


## Commands

These are the list of commands to be executed. The notebook calls the actual train.py with the appropriate cli arg.

In [2]:
commands = [
    "uv sync --extra train",
    "python3 training/train.py sanity-check",
    "python3 training/train.py export-sft --output data/sft_chat_examples.jsonl",
    "python3 training/train.py train-sft --output-dir training_outputs/sft_warm_start",
    "python3 training/train.py train-grpo --model-name training_outputs/sft_warm_start --output-dir training_outputs",
    "python3 training/demo_comparison.py --post-sft-dir training_outputs/sft_warm_start --post-rl-dir training_outputs/final_merged --output training_outputs/test_eval_metrics.json",
    "python3 training/plot_rewards.py --reward-input training_outputs/reward_history.json --eval-input training_outputs/test_eval_metrics.json --output training_outputs/training_and_eval_report.png",
]

for idx, cmd in enumerate(commands, start=1):
    print(f"{idx}. {cmd}")


1. uv sync --extra train
2. python3 training/train.py sanity-check
3. python3 training/train.py export-sft --output data/sft_chat_examples.jsonl
4. python3 training/train.py train-sft --output-dir training_outputs/sft_warm_start
5. python3 training/train.py train-grpo --model-name training_outputs/sft_warm_start --output-dir training_outputs
6. python3 training/demo_comparison.py --post-sft-dir training_outputs/sft_warm_start --post-rl-dir training_outputs/final_merged --output training_outputs/test_eval_metrics.json
7. python3 training/plot_rewards.py --reward-input training_outputs/reward_history.json --eval-input training_outputs/test_eval_metrics.json --output training_outputs/training_and_eval_report.png


## 1. Install Training Dependencies

In [ ]:
run("uv sync --extra train")

## 2. Optional Sanity Check

This confirms the environment can execute the training entrypoint before spending GPU time on SFT or RL.

In [ ]:
run("python3 training/train.py sanity-check")

## 3. Export SFT Examples

In [ ]:
run("python3 training/train.py export-sft --output data/sft_chat_examples.jsonl")

## 4. Run SFT

In [ ]:
run("python3 training/train.py train-sft --output-dir training_outputs/sft_warm_start")

## 5. Run RL From The SFT Checkpoint

In [ ]:
run("python3 training/train.py train-grpo --model-name training_outputs/sft_warm_start --output-dir training_outputs")

## 6. Evaluate Baseline vs Post-SFT vs Post-RL

In [ ]:
run("python3 training/demo_comparison.py --post-sft-dir training_outputs/sft_warm_start --post-rl-dir training_outputs/final_merged --output training_outputs/test_eval_metrics.json")

## 7. Plot Reward Curve And Final Eval Metrics

In [ ]:
run("python3 training/plot_rewards.py --reward-input training_outputs/reward_history.json --eval-input training_outputs/test_eval_metrics.json --output training_outputs/training_and_eval_report.png")

## Artifact Summary

These cells make the story visible after the run finishes.

In [ ]:
for path in [SFT_DIR, FINAL_DIR, REWARD_HISTORY, EVAL_METRICS, REPORT_IMAGE]:
    print(f"{path}: {'present' if path.exists() else 'missing'}")


In [ ]:
show_reward_summary(REWARD_HISTORY)

In [ ]:
show_eval_table(EVAL_METRICS)

In [ ]:
display(Image(filename=str(REPORT_IMAGE)))